In [2]:
class Resistor(object):
    def __init__(self, ohms):
        self.ohms = ohms
        self.voltage = 0
        self.current = 0
        
r1 = Resistor(50e3)
r1.ohms = 10e3


class VoltageResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)
        self._voltage = 0
        
    @property
    def voltage(self):
        return self._voltage
    
    @voltage.setter
    def voltage(self, voltage):
        self._voltage = voltage
        self.current = self._voltage /self.ohms
        
r2 = VoltageResistance(1e3)
print('Before :%5r amps' % r2.current)
r2.voltage = 10
print('After : %5r amps'% r2.current)

Before :    0 amps
After :  0.01 amps


In [3]:
class BoundedResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)
        
    @property
    def ohms(self):
        return self._ohms

    @ohms.setter
    def ohms(self, ohms):
        if ohms <= 0:
            raise ValueError('%f ohms must be > 0' % ohms)
        self._ohms = ohms
    
r3 =BoundedResistance(1e3)
r3.ohms = 0

ValueError: 0.000000 ohms must be > 0

In [5]:
class FixedResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)
        
    @property
    def ohms(self):
        return self._ohms

    @ohms.setter
    def ohms(self, ohms):
        if hasattr(self, '_ohms'):
            raise AttributeError("Can't set attribute")
        self._ohms=ohms
        
r4 = FixedResistance(1e3)
r4.ohms = 2e3


AttributeError: Can't set attribute

In [11]:
import time
from datetime import datetime, timedelta

class Bucket(object):
    def __init__(self, period):
        self.period_data = timedelta(seconds=period)
        self.reset_time = datetime.now()
        self.quota = 0
        
    def __repr__(self):
        return 'Bucket(quota=%d)' % self.quota
        
    
    # 채우는 동작의 구현
    def fill(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            bucket.quota = 0
            bucket.reset_time = now
        bucket.quota += amount
    # 할당량 소비의 구현
    def deduct(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            return False
        if bucket.quota - amount < 0 :
            return False
        bucket.quota -= amount
        return True
 # 양동이 채우기   
bucket = Bucket(60)
bucket.fill(100)
print(bucket)

if bucket.deduct(99):
    print('Had 99 quota')
else:
    print('Not enough for 99 quota')
print(bucket)    

if bucket.deduct(3):
    print('Had 3 quota')
else:
    print('Not enough for 3 quota')
print(bucket)    

Bucket(quota=100)
Had 99 quota
Bucket(quota=1)
Not enough for 3 quota
Bucket(quota=1)


In [13]:
import time
from datetime import datetime, timedelta

class Bucket(object):
    def __init__(self, period):
        self.period_data = timedelta(seconds=period)
        self.reset_time = datetime.now()
        self.max_quota = 0
        self.quota_consumed = 0
        
    def __repr__(self):
        return 'Bucket(quota=%d, qutoa consumed=%d)' % (self.max_quota, self.quota_consumed)
    
    @property
    def quota(self):
        return self.max_quota - self.quota_consumed
    
    @quota.setter
    def quota(self, amount):
        delta = self.max_quota - amount
        if amount == 0:
            #새 기간의 할당량을 리셋함
            self.quota_consumed = 0
            self.max_quota = 0
        elif delta < 0:
            # 새 기간의 할당량을 채움
            assert self.quota_consumed == 0
            self.max_quota = amount
        else:
            # 기간동안 할당량의 소비
            assert self.max_quota >= self.quota_consumed
            self.quota_consumed += delta
            
    def fill(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            bucket.quota = 0
            bucket.reset_time = now
        bucket.quota += amount
    # 할당량 소비의 구현
    def deduct(bucket, amount):
        now = datetime.now()
        if now - bucket.reset_time > bucket.period_data:
            return False
        if bucket.quota - amount < 0 :
            return False
        bucket.quota -= amount
        return True
        
    

 # 양동이 채우기   
bucket = Bucket(60)
bucket.fill(100)
print(bucket)

if bucket.deduct(99):
    print('Had 99 quota')
else:
    print('Not enough for 99 quota')
print(bucket)    

if bucket.deduct(3):
    print('Had 3 quota')
else:
    print('Not enough for 3 quota')
print(bucket)    

Bucket(quota=100, qutoa consumed=0)
Had 99 quota
Bucket(quota=100, qutoa consumed=99)
Not enough for 3 quota
Bucket(quota=100, qutoa consumed=99)


In [15]:
class Homework(object):
    def __init__(self):
        self._grade = 0
    
    @property
    def grade(self):
        return self._grade

    @grade.setter
    def grade(self, value):
        if not(0 <= value <= 100):
            raise ValueError('Grade must be between 0 and 100')
        self._grade = value
        
galic = Homework()
galic.grade = 95
print(galic)


In [ ]:
from weakref import WeakKeyDictionary

class Grade(object):
    def __init__(self):
        self._value = WeakKeyDictionary()
        
    def __get__(self, instance, instance_type):
        if instance is None : return self
        return self._value.get(instance, 0)
    
    def __set__(self, instance, value):
        if not (0 <= value <= 100):
            raise ValueError('Grade must be between 0 and 100')
        self._value[instance] = value

class Exam(object):
    math_grade = Grade()
    writing_grade = Grade()
    science_grade = Grade()
        
first_exam = Exam()
first_exam.writing_grade = 82
first_exam.science_grade = 99
print(first_exam.writing_grade,first_exam.science_grade )

second_exam = Exam()
second_exam.writing_grade = 75
print(first_exam.writing_grade,second_exam.writing_grade )

82 99
75 0
